***Proyecto 08:*** Chatbot con Historial usando la API de Gemini + ipywidgets

***Descripción del demo:***
Este proyecto implementa un chatbot conversacional con memoria de historial utilizando la API de Google Gemini.
La interfaz interactiva está construida con `ipywidgets` directamente dentro del notebook, permitiendo al usuario
chatear con la IA sin salir del entorno de Colab. El modelo recuerda el contexto de toda la conversación.

***Tecnologías:***

| Herramienta                | Uso principal                                              |
|----------------------------|------------------------------------------------------------|  
| **Python 3**               | Lenguaje principal de desarrollo                           |
| **google-generativeai**    | SDK oficial de Google para acceder a la API de Gemini      |
| **ipywidgets**             | Creación de interfaz interactiva dentro del notebook       |
| **IPython.display**        | Renderizado de widgets y salida enriquecida en Colab       |
| **Google Colab**           | Entorno de ejecución recomendado en la nube                |

# ***Demo***

🔑 **PASO PREVIO: Obtén tu API Key gratis en** [https://aistudio.google.com](https://aistudio.google.com) → *Get API Key*

🔧 INSTALACIÓN DE LIBRERÍAS

In [14]:
# Instalar la librería oficial de Google para usar la API de Gemini
!pip install -q google-generativeai

# Instalar ipywidgets para crear la interfaz interactiva dentro del notebook
!pip install -q ipywidgets

# Activar extensión de ipywidgets en Colab (necesario para renderizar los widgets)
from google.colab import output
output.enable_custom_widget_manager()

print("✅ Librerías instaladas correctamente")

✅ Librerías instaladas correctamente


📦 IMPORTACIÓN DE LIBRERÍAS

In [15]:
# Librería oficial de Google para interactuar con la API de Gemini
import google.generativeai as genai

# ipywidgets para construir la interfaz de chat (cuadro de texto, botones, área de salida)
import ipywidgets as widgets

# display: para mostrar los widgets en el notebook | clear_output: para redibujar el chat
from IPython.display import display, clear_output, HTML

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


🔑 CONFIGURACIÓN DE API KEY Y MODELO

In [16]:
# ⚠️  REEMPLAZA este valor con tu API Key real de Google AI Studio
API_KEY = "TU_API_KEY_AQUI"

# Autenticar la aplicación con la clave de acceso a la API de Gemini
genai.configure(api_key=API_KEY)

# Inicializar el modelo Gemini 1.5 Flash (rápido, gratuito y con alta capacidad de contexto)
modelo = genai.GenerativeModel("gemini-2.0-flash")

print("✅ Modelo Gemini configurado correctamente")
print("📌 Modelo en uso: gemini-2.0-flash")

✅ Modelo Gemini configurado correctamente
📌 Modelo en uso: gemini-1.5-flash


💬 CONSTRUCCIÓN DEL CHATBOT CON HISTORIAL

In [19]:
# Lista global que almacena todos los mensajes intercambiados en la conversación
historial_conversacion = []

# ── Función principal: envía el mensaje del usuario y obtiene la respuesta de Gemini ──
import time

def enviar_mensaje(evento):
    mensaje_usuario = entrada_texto.value.strip()
    if not mensaje_usuario:
        return

    # Agregar mensaje del usuario al historial
    historial_conversacion.append({
        "role": "user",
        "parts": [mensaje_usuario]
    })

    entrada_texto.value = ""        # Limpiar input
    actualizar_pantalla_chat()      # Mostrar mensaje del usuario ANTES de esperar respuesta

    respuesta_texto = None

    try:
        sesion_chat = modelo.start_chat(history=historial_conversacion[:-1])
        respuesta = sesion_chat.send_message(mensaje_usuario)
        respuesta_texto = respuesta.text

    except Exception as error:
        if "429" in str(error):
            # Mostrar aviso de espera en pantalla
            historial_conversacion.append({
                "role": "model",
                "parts": ["⏳ Límite de API alcanzado. Reintentando en 15 segundos..."]
            })
            actualizar_pantalla_chat()
            time.sleep(15)

            # Quitar el mensaje de espera antes de reintentar
            historial_conversacion.pop()

            try:
                sesion_chat = modelo.start_chat(history=historial_conversacion[:-1])
                respuesta = sesion_chat.send_message(mensaje_usuario)
                respuesta_texto = respuesta.text
            except:
                respuesta_texto = "⚠️ Límite de API alcanzado. Espera unos segundos e intenta de nuevo."
        else:
            respuesta_texto = f"❌ Error: {str(error)}"

    # Agregar respuesta final al historial
    historial_conversacion.append({
        "role": "model",
        "parts": [respuesta_texto]
    })

    actualizar_pantalla_chat()

# ── Función para dibujar todos los mensajes del historial en el área de chat ──
def actualizar_pantalla_chat():
    with area_visualizacion:             # Trabajar dentro del widget de salida
        clear_output(wait=True)          # Borrar el contenido anterior para redibujar limpiamente
        for mensaje in historial_conversacion:           # Recorrer cada mensaje del historial
            if mensaje["role"] == "user":                # Si el mensaje es del usuario
                print(f"🧑 Tú:     {mensaje['parts'][0]}")    # Mostrarlo con ícono de persona
                print("─" * 60)                               # Separador visual
            else:                                        # Si el mensaje es del modelo
                print(f"🤖 Gemini: {mensaje['parts'][0]}")    # Mostrarlo con ícono de robot
                print("─" * 60)                               # Separador visual


# ── Función para borrar el historial y comenzar una conversación nueva ──
def limpiar_historial(evento):
    global historial_conversacion        # Acceder a la variable global del historial
    historial_conversacion = []          # Reiniciar la lista vaciándola completamente
    with area_visualizacion:             # Limpiar también la pantalla visual
        clear_output()
    print("✅ Historial limpiado. Puedes iniciar una nueva conversación.")


# ── Construcción de los elementos visuales de la interfaz ──

# Campo de texto donde el usuario escribe sus mensajes
entrada_texto = widgets.Text(
    placeholder="Escribe tu mensaje y presiona Enviar...",    # Texto de sugerencia
    layout=widgets.Layout(width="74%", height="35px")         # Tamaño del campo
)

# Botón azul para enviar el mensaje al chatbot
boton_enviar = widgets.Button(
    description="Enviar ➤",             # Texto visible en el botón
    button_style="primary",              # Estilo: azul (primario)
    layout=widgets.Layout(width="13%")   # Ancho del botón
)

# Botón naranja para limpiar el historial y reiniciar la conversación
boton_limpiar = widgets.Button(
    description="🗑️ Nuevo chat",         # Texto visible en el botón
    button_style="warning",              # Estilo: naranja (advertencia)
    layout=widgets.Layout(width="13%")   # Ancho del botón
)

# Área donde se muestran los mensajes del chat (con borde y scroll)
area_visualizacion = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #cccccc",      # Borde gris alrededor del área
        min_height="350px",              # Altura mínima del área de chat
        max_height="500px",              # Altura máxima antes de hacer scroll
        padding="12px",                  # Espacio interno del área
        overflow_y="auto"                # Activar scroll vertical automático
    )
)

# Conectar los botones con sus respectivas funciones
boton_enviar.on_click(enviar_mensaje)        # Al hacer clic en Enviar → ejecutar enviar_mensaje
boton_limpiar.on_click(limpiar_historial)    # Al hacer clic en Limpiar → ejecutar limpiar_historial

# Fila inferior que agrupa el campo de texto y los dos botones
fila_controles = widgets.HBox(
    [entrada_texto, boton_enviar, boton_limpiar],
    layout=widgets.Layout(gap="8px", margin="8px 0 0 0")   # Espacio entre elementos
)

# Ensamblar y mostrar la interfaz completa
display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 8px 0'>💬 Chatbot con Gemini AI — Historial de conversación</h3>"),
    area_visualizacion,    # Área donde se ven los mensajes
    fila_controles         # Controles de entrada (texto + botones)
]))
print("✅ Chatbot listo. Escribe tu primer mensaje para comenzar.")

✅ Chatbot listo. Escribe tu primer mensaje para comenzar.


📘 NOTAS Y SOLUCIÓN DE PROBLEMAS

In [18]:
# ── Información útil para quienes ejecuten este notebook ──

print("📌 INSTRUCCIONES DE USO:")
print("   1. Obtén tu API Key gratuita en: https://aistudio.google.com")
print("   2. Reemplaza 'TU_API_KEY_AQUI' en la celda de configuración")
print("   3. Ejecuta todas las celdas en orden (Entorno de ejecución → Ejecutar todo)")
print("   4. Escribe tu mensaje en el campo de texto y presiona 'Enviar'")
print()
print("⚠️  SOLUCIÓN DE PROBLEMAS:")
print("   • Error 'API_KEY_INVALID': Verifica que tu API key sea correcta")
print("   • Widgets no aparecen: Ejecuta la celda de instalación nuevamente")
print("   • Error de cuota: El plan gratuito tiene límite de solicitudes por minuto")
print()
print("🔒 SEGURIDAD: Nunca subas tu API Key a GitHub.")
print("   Usa los 'Secrets' de Colab: ícono de llave 🔑 en el panel izquierdo.")

📌 INSTRUCCIONES DE USO:
   1. Obtén tu API Key gratuita en: https://aistudio.google.com
   2. Reemplaza 'TU_API_KEY_AQUI' en la celda de configuración
   3. Ejecuta todas las celdas en orden (Entorno de ejecución → Ejecutar todo)
   4. Escribe tu mensaje en el campo de texto y presiona 'Enviar'

⚠️  SOLUCIÓN DE PROBLEMAS:
   • Error 'API_KEY_INVALID': Verifica que tu API key sea correcta
   • Widgets no aparecen: Ejecuta la celda de instalación nuevamente
   • Error de cuota: El plan gratuito tiene límite de solicitudes por minuto

🔒 SEGURIDAD: Nunca subas tu API Key a GitHub.
   Usa los 'Secrets' de Colab: ícono de llave 🔑 en el panel izquierdo.
